In [7]:


import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-51-20-128-41.eu-north-1.compute.amazonaws.com:5000/")

In [8]:
# set or create an experiment
mlflow.set_experiment("exp9_sbert_lightgbm_hybrid") 


<Experiment: artifact_location='s3://my-s3-bucket-of-store-artifact-youtube-data12/mlflow-artifacts/13', creation_time=1764681064378, experiment_id='13', last_update_time=1764681064378, lifecycle_stage='active', name='exp9_sbert_lightgbm_hybrid', tags={}>

In [9]:
import pandas as pd
df=pd.read_csv('sentiment_clean.csv') 

In [10]:
df['sentiment_numeric']=df.pop('sentiment_numeric')

In [11]:
df

,text_clean,word_count,num_stop_words,num_chars,num_punctuation_chars,category_gaming,category_movies,category_music,category_technology,hour,...,anger,anticipation,trust,surprise,positive,negative,sadness,disgust,joy,sentiment_numeric
0,all products can be found on since i review 5...,24,9,116,1,0.0,0.0,0.0,1.0,19,...,0.0,0.0,0.333333,0.0,0.333333,0.0,0.0,0.0,0.333333,1
1,bro how to talk to woman in 6 steps is so rela...,12,5,53,0,0.0,0.0,0.0,1.0,23,...,0.0,0.0,0.000000,0.0,1.000000,0.0,0.0,0.0,0.000000,0
2,i was gonna say does it give you the drinks fo...,12,7,54,1,0.0,0.0,0.0,1.0,16,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0
3,anyone gonna talk abt what was o. his pc,9,3,40,1,0.0,0.0,0.0,1.0,22,...,0.0,0.0,0.000000,0.0,1.000000,0.0,0.0,0.0,0.000000,0
4,how is everyone not talking about his search?!...,15,6,85,4,0.0,0.0,0.0,1.0,12,...,0.0,0.0,0.000000,0.0,0.500000,0.0,0.0,0.0,0.500000,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49472,ye त बककफ शकष मतर ह,6,0,30,0,0.0,0.0,0.0,0.0,9,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0
49473,enke baat ka koi bharosa nahi hai.,7,0,34,1,0.0,0.0,0.0,0.0,9,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0
49474,15 september tak 1 lakh notification nikalo,8,0,45,0,0.0,0.0,0.0,0.0,9,...,0.0,1.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0
49475,1st last education minister in bihar,7,1,38,0,0.0,0.0,0.0,0.0,9,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0


In [12]:

import os
import numpy as np
import joblib
import mlflow
import optuna
import lightgbm as lgb
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV

from imblearn.over_sampling import SMOTE
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# -------------------------
# CONFIG
# -------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS = 3
N_TRIALS = 25
EARLY_STOPPING_ROUNDS = 50
MLFLOW_EXPERIMENT_NAME = "Exp9_SBERT_LightGBM_HYBRID_FIXED"
SBERT_MODEL = "all-MiniLM-L6-v2"
TF_BATCH_SIZE = 256
N_JOBS = -1

MODEL_DIR = "models/"
MLFLOW_TRACKING_URI = "mlruns/"
ARTIFACT_DIR = "./artifacts/"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# -------------------------
# Load df OR assume it's in memory
# -------------------------
# If df not in memory, uncomment and adjust path:
# df = pd.read_csv("data/processed/features.csv")

# ensure df exists
if "df" not in globals():
    raise RuntimeError("DataFrame `df` not found in memory. Load it or run in notebook with df defined.")

# -------------------------
# Target mapping & cleanup
# -------------------------
df['sentiment_numeric'] = df['sentiment_numeric'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['text_clean', 'sentiment_numeric']).reset_index(drop=True)
y_all = df['sentiment_numeric'].astype(int).to_numpy()

# -------------------------
# Numeric features (same behaviour as before)
# -------------------------
if df.shape[1] > 2:
    X_numeric = df.iloc[:, 1:-1]
    scaler = StandardScaler(with_mean=False)
    X_numeric_scaled = scaler.fit_transform(X_numeric)
    has_numeric = True
else:
    X_numeric_scaled = None
    has_numeric = False

# -------------------------
# Create SBERT embeddings
# -------------------------
print("Loading SBERT model:", SBERT_MODEL)
sbert = SentenceTransformer(SBERT_MODEL)

texts = df['text_clean'].astype(str).tolist()
embeddings = []
print("Creating SBERT embeddings (batches):")
for i in tqdm(range(0, len(texts), TF_BATCH_SIZE)):
    batch = texts[i:i+TF_BATCH_SIZE]
    emb = sbert.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    embeddings.append(emb)
embeddings = np.vstack(embeddings)
print("Embeddings shape:", embeddings.shape)

# -------------------------
# Combine SBERT + numeric features
# -------------------------
if has_numeric:
    X_numeric_arr = np.asarray(X_numeric_scaled)
    X = np.hstack([embeddings, X_numeric_arr])
else:
    X = embeddings

print("Final feature shape:", X.shape)

# -------------------------
# Train/test split (stratified)
# -------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)

print("Train distribution (before any balancing):", dict(zip(*np.unique(y_train_full, return_counts=True))))
print("Test distribution:", dict(zip(*np.unique(y_test, return_counts=True))))

# -------------------------
# IMPORTANT: compute class weights from ORIGINAL training distribution (before SMOTE)
# -------------------------
classes = np.unique(y_train_full)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, cw)}
print("Class weights (from original train):", class_weight_dict)

# -------------------------
# Apply SMOTE on training set to reduce class imbalance (if needed)
# Use SMOTE to produce a balanced training set for training and CV.
# -------------------------
print("Applying SMOTE to training set (k_neighbors=5)...")
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_bal, y_train_bal = sm.fit_resample(X_train_full, y_train_full)
print("Train distribution (after SMOTE):", dict(zip(*np.unique(y_train_bal, return_counts=True))))

# Convert to numpy arrays (safe indexing for CV)
X_train_bal = np.asarray(X_train_bal)
y_train_bal = np.asarray(y_train_bal)
X_test = np.asarray(X_test)
y_test = np.asarray(y_test)

# compute sample_weight for final training from original class weights (not from SMOTE counts)
# This keeps class weighting aligned to original distribution while using SMOTE data to expose minority class
sample_weight_full = np.array([class_weight_dict[int(lbl)] for lbl in y_train_bal])

# -------------------------
# Optuna objective (uses StratifiedKFold CV on balanced train)
# maximize macro recall
# -------------------------
def objective(trial):
    params = {
        "boosting_type": "gbdt",
        "objective": "multiclass",
        "num_class": len(classes),
        "metric": "multi_logloss",
        "n_jobs": N_JOBS,
        "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 2.0),
        "class_weight": class_weight_dict,
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "random_state": RANDOM_STATE
    }

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    recalls = []

    # Use numpy indices from balanced train to avoid pandas KeyError
    for train_idx, val_idx in skf.split(X_train_bal, y_train_bal):
        X_tr, X_val = X_train_bal[train_idx], X_train_bal[val_idx]
        y_tr, y_val = y_train_bal[train_idx], y_train_bal[val_idx]

        # sample weights for this fold — based on original class weights
        sw_tr = np.array([class_weight_dict[int(lbl)] for lbl in y_tr])

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)]
        )

        preds = model.predict(X_val)
        r = recall_score(y_val, preds, average="macro")
        recalls.append(r)

    return float(np.mean(recalls))

# run study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best optuna params:", study.best_params)

# -------------------------
# Final train on full balanced training set with best params (and original class weights)
# -------------------------
best_params = study.best_params.copy()
best_params.update({
    "boosting_type": "gbdt",
    "objective": "multiclass",
    "num_class": len(classes),
    "metric": "multi_logloss",
    "class_weight": class_weight_dict,
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
    "verbosity": -1
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(
    X_train_bal, y_train_bal,
    sample_weight=sample_weight_full,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS)]
)

# -------------------------
# Calibrate probabilities using a small calibration split drawn from the balanced train
# (we'll create the holdout now, so calibration uses unseen examples)
# -------------------------
from sklearn.model_selection import train_test_split as sk_train_test_split
X_model_train, X_holdout_cal, y_model_train, y_holdout_cal, sw_model_train, sw_holdout = sk_train_test_split(
    X_train_bal, y_train_bal, sample_weight_full, test_size=0.10, random_state=RANDOM_STATE, stratify=y_train_bal
)

# re-fit a fresh model on X_model_train (with best_params) for calibration's 'prefit' style
cal_model = lgb.LGBMClassifier(**best_params)
cal_model.fit(X_model_train, y_model_train, sample_weight=np.array(sw_model_train),
              eval_set=[(X_holdout_cal, y_holdout_cal)],
              eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS)])

print("Calibrating probabilities using holdout calibration set (sigmoid)...")
calibrator = CalibratedClassifierCV(estimator=cal_model, method='sigmoid', cv='prefit')
calibrator.fit(X_holdout_cal, y_holdout_cal)

# -------------------------
# Threshold sweep on holdout to pick CONF_THRESH
# -------------------------
probs_holdout = calibrator.predict_proba(X_holdout_cal)

def apply_conf_thresh(probs, thresh):
    preds = np.argmax(probs, axis=1)
    low_conf = probs.max(axis=1) < thresh
    preds[low_conf] = 0  # map low-confidence to Neutral class (0) *optional*
    return preds

threshs = np.linspace(0.30, 0.60, 31)
best_t = 0.45
best_score = -1
for t in threshs:
    p = apply_conf_thresh(probs_holdout.copy(), t)
    sc = recall_score(y_holdout_cal, p, average='macro')
    if sc > best_score:
        best_score = sc
        best_t = t

CONF_THRESH = float(best_t)
print("Best threshold on holdout:", CONF_THRESH, "holdout macro recall:", best_score)

# -------------------------
# Evaluate on test set (calibrated probabilities + threshold)
# -------------------------
probs_test = calibrator.predict_proba(X_test)
y_pred = np.argmax(probs_test, axis=1)
low_conf_mask = probs_test.max(axis=1) < CONF_THRESH
# NOTE: here we keep the low-confidence -> Neutral behavior because your pipeline used it.
# If you prefer pure predictions, comment out the next line.
y_pred[low_conf_mask] = 0

acc = accuracy_score(y_test, y_pred)
macro_rec = recall_score(y_test, y_pred, average="macro")
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nFINAL RESULTS (Exp9 )")
print("Accuracy:", acc)
print("Macro Recall:", macro_rec)
print(report)
print("Confusion Matrix:\n", cm)

# -------------------------
# MLflow logging & saving artifacts
# -------------------------
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
with mlflow.start_run(run_name="Exp9_SBERT_LGBM_HYBRID"):
    mlflow.log_params(best_params)
    mlflow.log_param("conf_threshold", CONF_THRESH)
    mlflow.log_metric("accuracy", float(acc))
    mlflow.log_metric("macro_recall", float(macro_rec))

    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix -  SBERT+LGBM (Exp9 )")
    plt.savefig("exp9_fixed_confusion_matrix.png")
    mlflow.log_artifact("exp9_fixed_confusion_matrix.png")
    plt.close()

    # save classification report
    with open("exp9_fixed_classification_report.txt", "w", encoding="utf8") as f:
        f.write(report)
    mlflow.log_artifact("exp9_fixed_classification_report.txt")

    # save model + sbert + scaler
    joblib.dump(calibrator, os.path.join(MODEL_DIR, "exp9_fixed_calibrated_model.pkl"))
    joblib.dump(sbert, os.path.join(MODEL_DIR, "exp9_fixed_sbert_model.pkl"))
    if has_numeric:
        joblib.dump(scaler, os.path.join(MODEL_DIR, "exp9_fixed_numeric_scaler.pkl"))

    mlflow.log_artifact(os.path.join(MODEL_DIR, "exp9_fixed_calibrated_model.pkl"))
    mlflow.log_artifact(os.path.join(MODEL_DIR, "exp9_fixed_sbert_model.pkl"))
    if has_numeric:
        mlflow.log_artifact(os.path.join(MODEL_DIR, "exp9_fixed_numeric_scaler.pkl"))

    # feature importances
    try:
        fi = cal_model.feature_importances_
        emb_dim = embeddings.shape[1]
        if has_numeric:
            numeric_names = list(X_numeric.columns)
        else:
            numeric_names = []
        feature_names = [f"sbert_{i}" for i in range(emb_dim)] + numeric_names
        fi_df = pd.DataFrame({"feature": feature_names, "importance": fi})
        fi_df = fi_df.sort_values("importance", ascending=False).reset_index(drop=True)
        fi_df.to_csv("exp9_fixed_feature_importances.csv", index=False)
        mlflow.log_artifact("exp9_fixed_feature_importances.csv")
    except Exception as e:
        print("Could not log feature importances:", e)

print("Artifacts saved to MLflow and", MODEL_DIR)
print("Done.")


Loading SBERT model: all-MiniLM-L6-v2
Creating SBERT embeddings (batches):


  0%|          | 0/194 [00:00<?, ?it/s]

Embeddings shape: (49477, 384)
Final feature shape: (49477, 405)
Train distribution (before any balancing): {np.int64(0): np.int64(18182), np.int64(1): np.int64(17949), np.int64(2): np.int64(3450)}
Test distribution: {np.int64(0): np.int64(4546), np.int64(1): np.int64(4488), np.int64(2): np.int64(862)}
Class weights (from original train): {0: 0.7256444102225644, 1: 0.7350641632774342, 2: 3.8242512077294686}
Applying SMOTE to training set (k_neighbors=5)...


[I 2025-12-03 11:36:15,756] A new study created in memory with name: no-name-08f00a69-5f9c-425c-bb43-7f499523f1be


Train distribution (after SMOTE): {np.int64(0): np.int64(18182), np.int64(1): np.int64(18182), np.int64(2): np.int64(18182)}


...

[I 2025-12-03 15:51:51,173] Trial 24 finished with value: 0.8154401828567613 and parameters: {'learning_rate': 0.03931430778088897, 'num_leaves': 108, 'max_depth': 16, 'min_child_samples': 153, 'feature_fraction': 0.7630073952370671, 'bagging_fraction': 0.6762952587557451, 'bagging_freq': 4, 'lambda_l1': 1.4024662481705903, 'lambda_l2': 0.9985028295916787, 'n_estimators': 713}. Best is trial 18 with value: 0.8182085298669008.
Best optuna params: {'learning_rate': 0.03487652275657492, 'num_leaves': 181, 'max_depth': 16, 'min_child_samples': 160, 'feature_fraction': 0.7804503427378252, 'bagging_fraction': 0.7051102112946549, 'bagging_freq': 4, 'lambda_l1': 1.340805141865139, 'lambda_l2': 1.2855309089718228, 'n_estimators': 889}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[672]	valid_0's multi_logloss: 0.593905
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[750]	valid_0's multi_logloss: 0

e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Best threshold on holdout: 0.52 holdout macro recall: 0.8381038030502702


e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



FINAL RESULTS (Exp9 )
Accuracy: 0.7549514955537591
Macro Recall: 0.7426754002809929
              precision    recall  f1-score   support

           0       0.73      0.79      0.76      4546
           1       0.81      0.73      0.77      4488
           2       0.63      0.71      0.67       862

    accuracy                           0.75      9896
   macro avg       0.73      0.74      0.73      9896
weighted avg       0.76      0.75      0.76      9896

Confusion Matrix:
 [[3605  680  261]
 [1141 3254   93]
 [ 188   62  612]]
Artifacts saved to MLflow and models/
Done.
